In [33]:
import pandas as pd
import numpy as np

# Clases propias
from etl import Dataloader
from feature_engineer import add_features
from train import Train
from train_with_mlflow import TrainWithMLflow

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from xgboost import XGBClassifier

# MLflow
import mlflow


ETL

In [34]:
loader = Dataloader(
    path_to_save=r"C:/Users/Valentina Molina/Documents/Repositorios/Proyecto_Final_MLOps/data/PS_20174392719_1491204439457_log.csv",
    n_samples=50000
)
df = loader.load_data()
loader.drop_name_columns()
df = loader.df.copy()

print(df.shape)
df.head()

(50000, 8)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0


Feature Engineering

In [35]:
df = add_features(df)
df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,diff_old_new_orig,diff_old_new_dest,amount_to_orig_balance,amount_to_dest_balance
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,9839.64,0.0,0.057834,9839.640000
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,1864.28,0.0,0.087731,1864.280000
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,181.00,0.0,0.994505,181.000000
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,181.00,21182.0,0.994505,0.008545
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,11668.14,0.0,0.280788,11668.140000


Definir variables

In [36]:
numeric_features = [
    'step', 'amount',
    'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',
    'diff_old_new_orig', 'diff_old_new_dest',
    'amount_to_orig_balance', 'amount_to_dest_balance'
]

categorical_features = ['type']
target_column = 'isFraud'
test_size = 0.2


Step 1: Modelando sin MLflow

In [37]:
# Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)
trainer = Train(df, numeric_features, categorical_features, target_column, model, test_size)
pipeline = trainer.train()
print("Modelo Logistic Regression entrenado sin MLflow ✅")

# Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
trainer = Train(df, numeric_features, categorical_features, target_column, model, test_size)
pipeline = trainer.train()
print("Modelo RandomForest entrenado sin MLflow ✅")


2025/10/02 20:51:22 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '4e78fb0dd1394a9ea27421b2b5c08bce', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run abundant-goat-760 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/4e78fb0dd1394a9ea27421b2b5c08bce
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877
Modelo Logistic Regression entrenado sin MLflow ✅


2025/10/02 20:51:42 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'f10680fab9034eba933527847ef44f5a', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run trusting-stag-234 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/f10680fab9034eba933527847ef44f5a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877
Modelo RandomForest entrenado sin MLflow ✅


Step 2: Modelando con MLflow (Logistic + RandomForest + LightGBM)

In [38]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraude-modelos")

mlflow.sklearn.autolog()

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

modelos = [
    ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=42)),
    ("RandomForest", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("LightGBM", lgb.LGBMClassifier(random_state=42))
]

In [39]:
resultados = {}
for nombre, modelo in modelos:
    print(f"\nEntrenando {nombre} con MLflow...")
    trainer = TrainWithMLflow(df, numeric_features, categorical_features, target_column, modelo, test_size)
    pipeline, run_id = trainer.train()
    resultados[nombre] = run_id

resultados


Entrenando LogisticRegression con MLflow...
MLflow Run ID: fc33f95d6c8646a68b5ab2b41e043e51
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 0.9984
Test Accuracy: 0.9978
🏃 View run dazzling-slug-759 at: http://127.0.0.1:5000/#/experiments/126333943536851071/runs/fc33f95d6c8646a68b5ab2b41e043e51
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/126333943536851071

Entrenando RandomForest con MLflow...
MLflow Run ID: 4502c117cfe141e59fe1c16780663a27
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 1.0000
Test Accuracy: 0.9997
🏃 View run gifted-moth-258 at: http://127.0.0.1:5000/#/experiments/126333943536851071/runs/4502c117cfe141e59fe1c16780663a27
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/126333943536851071

Entrenando LightGBM con MLflow...
[LightGBM] [Info] Number of positive: 80, number of negative: 39920
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006345 seconds.
You can set `force_row_wise=true` to remove the

{'LogisticRegression': 'fc33f95d6c8646a68b5ab2b41e043e51',
 'RandomForest': '4502c117cfe141e59fe1c16780663a27',
 'LightGBM': 'bb1e369a326d4c00bb012383e8a1400f'}

Modelo con XGBoost + MLflow

In [40]:
mlflow.set_experiment("fraude-xgboost")
mlflow.xgboost.autolog()

params_xgb = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8
}

model_xgb = XGBClassifier(**params_xgb, random_state=42, use_label_encoder=False, eval_metric="logloss")

trainer = TrainWithMLflow(df, numeric_features, categorical_features, target_column, model_xgb, test_size, params_xgb, mlflow)
pipeline, run_id = trainer.train()
print("Modelo XGBoost entrenado con MLflow ✅", run_id)


MLflow Run ID: 78ef18619fdf46bbb9ac0acb9cdd14ed
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 1.0000
Test Accuracy: 0.9994
🏃 View run unequaled-tern-284 at: http://127.0.0.1:5000/#/experiments/670538311910718369/runs/78ef18619fdf46bbb9ac0acb9cdd14ed
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/670538311910718369
Modelo XGBoost entrenado con MLflow ✅ 78ef18619fdf46bbb9ac0acb9cdd14ed


Step 3: Modelo con MLflow + Optuna

In [41]:
from train_with_mlflow_optuna import TrainWithMLflowOptuna

mlflow.set_experiment("fraude-optuna")

param_distributions = {
    'n_estimators': ('int', 50, 200),
    'max_depth': ('int', 5, 30),
    'min_samples_split': ('int', 2, 10),
    'min_samples_leaf': ('int', 1, 5),
    'max_features': ('categorical', ['sqrt', 'log2', None])
}

trainer = TrainWithMLflowOptuna(
    df=df,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    target_column=target_column,
    model_class=RandomForestClassifier,
    test_size=0.2,
    n_trials=20,
    optimization_metric='f1',
    param_distributions=param_distributions,
    model_params={'random_state': 42},
    mlflow_setup=mlflow
)

best_pipeline, run_id, study = trainer.train()
print("Mejor modelo con Optuna:", run_id)


[I 2025-10-02 20:53:40,367] A new study created in memory with name: no-name-67dcab82-e79e-4a0e-813d-47398b4c3955
2025/10/02 20:53:40 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'f5e6121f7f574c31b5c2e5b748f281b8', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run popular-asp-875 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/f5e6121f7f574c31b5c2e5b748f281b8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877


[I 2025-10-02 20:54:13,988] Trial 0 finished with value: 0.918918918918919 and parameters: {'n_estimators': 198, 'max_depth': 22, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.918918918918919.
2025/10/02 20:54:14 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '6ce8807d06c04eef9d61926f6e4347ab', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run blushing-skink-784 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/6ce8807d06c04eef9d61926f6e4347ab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877


[I 2025-10-02 20:55:36,802] Trial 1 finished with value: 0.6857142857142857 and parameters: {'n_estimators': 62, 'max_depth': 18, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': None}. Best is trial 0 with value: 0.918918918918919.
2025/10/02 20:55:36 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '8d9ee28321b1423caf036d413008dbaf', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


...

🏃 View run brawny-deer-98 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/d607345e4bf44d0ba9bac1f2faa094b2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877


[I 2025-10-02 21:05:17,114] Trial 18 finished with value: 0.6857142857142857 and parameters: {'n_estimators': 157, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 0 with value: 0.918918918918919.
2025/10/02 21:05:17 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '2676427b1a334af4b3d675f1955947f1', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run nervous-jay-466 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/2676427b1a334af4b3d675f1955947f1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877


[I 2025-10-02 21:05:41,027] Trial 19 finished with value: 0.5185185185185185 and parameters: {'n_estimators': 184, 'max_depth': 6, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.918918918918919.
2025/10/02 21:05:41 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '0f3838296cf24bd6bd776d85e8862787', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run respected-fowl-510 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/0f3838296cf24bd6bd776d85e8862787
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877
Optuna best params: {'random_state': 42, 'n_estimators': 198, 'max_depth': 22, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt'}
MLflow run_id: 015fba471e7e41ae9c1b76654571eaaa
🏃 View run grandiose-midge-813 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/015fba471e7e41ae9c1b76654571eaaa
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877
Mejor modelo con Optuna: 015fba471e7e41ae9c1b76654571eaaa
